# RQ1 factor C - reranking

**What varies:** whether a cross-encoder reorders the candidate list, two levels.
**What is held constant:** 512-unit sliding-window segments, BGE-small dense retrieval for
the candidate list, candidate depth 20, context depth 5, all 910 questions.

**The centre point.** Every notebook varies one factor and holds the rest at 512-unit sliding
window segments with 10% overlap, BGE-small dense retrieval, no reranking, candidate depth 20
and context depth 5. That is what makes each measurement a *main effect*: no arm differs from
the centre in two ways at once.

**Levels:** reranking off *(centre point)*, and `BAAI/bge-reranker-base` over the top 20
dense candidates.

Two optional extras, both off by default: reranking at depth 50, and reranking on top of the
hybrid list rather than the dense one. Each is a second factor, so neither belongs in the
main-effect estimate; they are reported separately as a sensitivity check.

### Why this is much faster than the earlier notebook
The cross-encoder is called **once** for the whole experiment, over every (query, segment)
pair at batch size 128 in half precision. Calling it per question on 20 pairs leaves the GPU
idle between calls; Table 5.2 recorded 435.69 seconds for that pattern on 450 questions.

**Runtime:** about 10 minutes for 910 questions x 20 segments.

## 1. Open the shared cache

In [ ]:
# Load the shared library written by notebook 00 and open the same cache.
import sys, os
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/techqa_rq1'
assert os.path.exists(PROJECT + '/rq1_core.py'), (
    'rq1_core.py not found. Run RQ1_00_Build_Once.ipynb first - it writes the library '
    'and builds the chunk tables and embeddings that this notebook reads from cache.')
sys.path.insert(0, PROJECT)
import rq1_core as C
import numpy as np, pandas as pd, time, json
C.setup(project_dir=PROJECT)
print('GPU:', C.gpu_name())

In [ ]:
# The whole labelled dataset, on the fast local disk.
# Notebook 00 mirrors the four extracted files to Drive, so this fresh runtime restores them
# with a file copy (~1 min) instead of re-downloading and re-extracting the archive (~5 min).
# getattr() keeps this working whichever build of notebook 00 wrote the library to Drive.
if not C.dataset_present():
    if getattr(C, 'dataset_mirrored', lambda: False)():
        print('restoring the corpus from the Drive mirror ...')
        C.restore_dataset_from_mirror()
    else:
        dl, ex = C.download_commands()
        get_ipython().system(dl)
        get_ipython().system(ex)
        if hasattr(C, 'mirror_dataset'):
            C.mirror_dataset()
ds = C.load_dataset()
print(json.dumps(ds.summary(), indent=1))

## 2. Configuration for this factor

In [ ]:
C.CFG.RERANK_DEPTH = 20
RUN_DEPTH_50    = False   # sensitivity check: rerank the top 50 instead of the top 20
RUN_ON_HYBRID   = False   # sensitivity check: rerank the hybrid-RRF list instead of the dense one
print('rerank depth', C.CFG.RERANK_DEPTH, '| model', C.CFG.RERANKER_MODEL)

## 3. Dense candidates, then one batched reranking pass

In [ ]:
idx = C.load_method_index(ds, C.CFG.BASELINE_METHOD, with_bm25=RUN_ON_HYBRID)
base = {q['QUESTION_ID']: idx.rank(q, 'dense') for q in ds.questions}
print(f'{len(base)} candidate lists ready')

from sentence_transformers import CrossEncoder
ce = CrossEncoder(C.CFG.RERANKER_MODEL, max_length=512, device=C.device())
if C.device() == 'cuda':
    ce.model.half()

reranked, rerank_s = C.rerank_batch(idx, ds.questions, base, depth=20, batch_size=128, model=ce)
n_pairs = sum(min(20, len(v)) for v in base.values())
print(f'{n_pairs:,} (query, segment) pairs scored in {rerank_s:.0f}s '
      f'= {1000*rerank_s/max(1,len(ds.questions)):.0f} ms per question')

In [ ]:
frames, contexts = [], {}
off, lat_off = C.run_arm(idx, ds.questions, 'dense', arm_name='off')
off['rerank_level'] = 'off'
on, _ = C.run_arm(idx, ds.questions, 'dense', reranked=reranked, arm_name='cross_encoder')
on['rerank_level'] = 'cross_encoder'
Cf = pd.concat([off, on], ignore_index=True)
contexts['off'] = {q['QUESTION_ID']: C.top_context(idx, q, 'dense').tolist()
                   for q in ds.questions}
contexts['cross_encoder'] = {qid: np.asarray(v)[:C.CFG.CONTEXT_K].tolist()
                             for qid, v in reranked.items()}
json.dump(contexts, open(C.PATHS.CACHE / 'contexts_factorC.json', 'w'))
C.save_table(Cf, 'rq1_factorC_per_question')
print('rows:', len(Cf))

## 4. The metric profile with and without reranking

In [ ]:
METRICS = C.RETRIEVAL_METRICS
summary_C = C.arm_summary(Cf, ['rerank_level'], METRICS)
summary_C['rerank_level'] = pd.Categorical(summary_C['rerank_level'],
                                           categories=['off', 'cross_encoder'], ordered=True)
summary_C = summary_C.sort_values('rerank_level')
display(summary_C[['rerank_level', 'n'] + METRICS].round(4))
C.save_table(summary_C, 'rq1_factorC_summary')
C.plot_metric_bars(summary_C, 'rerank_level', METRICS,
    'Factor C: cross-encoder reranking over dense candidates', 'fig_rq1_factorC_profile',
    baseline='off',
    caption='Retrieval-evidence profile with and without cross-encoder reranking of the top '
            '20 dense candidates, over all 610 answerable TechQA questions. Error bars are '
            '95% bootstrap intervals over questions.')

In [ ]:
contrasts_C = C.pairwise_against_baseline(Cf, 'rerank_level', 'off', METRICS)
display(contrasts_C[['metric', 'mean_diff', 'lo', 'hi', 'p_holm', 'significant', 'n']]
        .round(4))
C.save_table(contrasts_C, 'rq1_factorC_contrasts')
for _, r in contrasts_C[contrasts_C.metric.isin(['nDCG@5', 'gold_span_coverage'])].iterrows():
    print('\n' + C.describe_contrast(r))

## 5. What reranking does and does not move

Reranking cannot put a document into the candidate pool, so Recall@20 is fixed by construction. Its only lever is promotion inside that pool. Splitting the questions by whether the gold document was already in the top five says whether it rescues questions or disturbs ones that were already right.

In [ ]:
w5 = C.to_wide(Cf, 'gold_document_in_top_5', 'rerank_level')
rescued = int(((w5.off == 0) & (w5.cross_encoder == 1)).sum())
broken  = int(((w5.off == 1) & (w5.cross_encoder == 0)).sum())
recall_fixed = bool(np.allclose(C.to_wide(Cf, 'Recall@20', 'rerank_level').off,
                                C.to_wide(Cf, 'Recall@20', 'rerank_level').cross_encoder))
effect = {'questions': int(len(w5)), 'rescued_into_context': rescued,
          'demoted_out_of_context': broken, 'net': rescued - broken,
          'recall_at_20_unchanged': recall_fixed,
          'rerank_seconds_total': rerank_s,
          'rerank_ms_per_question': 1000 * rerank_s / max(1, len(ds.questions))}
print(json.dumps(effect, indent=1))
C.save_results(effect, 'rq1_factorC_promotion')

## 6. Sensitivity checks (optional)

Each of these changes a second thing, so they are reported apart from the main effect.

In [ ]:
extras = []
if RUN_DEPTH_50:
    rr50, s50 = C.rerank_batch(idx, ds.questions, base, depth=50, batch_size=128, model=ce)
    d50, _ = C.run_arm(idx, ds.questions, 'dense', reranked=rr50, arm_name='cross_encoder_d50')
    d50['rerank_level'] = 'cross_encoder_d50'
    extras.append(d50)
    print(f'depth 50: {s50:.0f}s')
if RUN_ON_HYBRID:
    hyb = {q['QUESTION_ID']: idx.rank(q, 'hybrid_rrf') for q in ds.questions}
    rrh, sh = C.rerank_batch(idx, ds.questions, hyb, depth=20, batch_size=128, model=ce)
    dh, _ = C.run_arm(idx, ds.questions, 'hybrid_rrf', reranked=rrh, arm_name='rerank_on_hybrid')
    dh['rerank_level'] = 'rerank_on_hybrid'
    extras.append(dh)
    print(f'on hybrid: {sh:.0f}s')
if extras:
    extra_df = pd.concat([Cf] + extras, ignore_index=True)
    s = C.arm_summary(extra_df, ['rerank_level'], METRICS)
    display(s[['rerank_level', 'n'] + METRICS].round(4))
    C.save_table(s, 'rq1_factorC_sensitivity')
else:
    print('sensitivity checks not run (both flags are False)')

## 7. What factor C contributes to RQ1

In [ ]:
del ce; C.free_gpu()
cost_C = pd.DataFrame([
    {'rerank_level': 'off', 'retrieval_ms': lat_off * 1000, 'rerank_ms': 0.0},
    {'rerank_level': 'cross_encoder', 'retrieval_ms': lat_off * 1000,
     'rerank_ms': 1000 * rerank_s / max(1, len(ds.questions))}])
cost_C['total_query_ms'] = cost_C.retrieval_ms + cost_C.rerank_ms
display(cost_C.round(2))
C.save_table(cost_C, 'rq1_factorC_cost')

eff_C = C.repeated_measures_effect(C.to_wide(Cf, 'nDCG@5', 'rerank_level'))
C.save_results({'effect_nDCG@5': eff_C, 'summary': summary_C.to_dict(orient='records'),
                'promotion': effect, 'cost': cost_C.to_dict(orient='records'),
                'provenance': C.stamp({'factor': 'reranking'})}, 'rq1_factorC')
print(f"Reranking, nDCG@5: levels span {eff_C['range']:.4f} "
      f"(95% CI {eff_C['range_lo']:.4f} to {eff_C['range_hi']:.4f}), "
      f"partial eta-squared {eff_C['eta2_partial']:.4f}, best level {eff_C['best_level']}.")
print(f"Cost of that: {cost_C.total_query_ms.iloc[1] / max(1e-9, cost_C.total_query_ms.iloc[0]):.0f}x "
      f"the query time of the centre point.")
print('\nFactor C complete. Run RQ1_D_Generation_Profile.ipynb next.')

## Worked example behind each result

Every table and figure above is an average over hundreds of questions. The cells below take **one real question** through the same calculation, so the reader can see how a single row becomes the number that is reported. Nothing is recomputed: the values are read back from the files this notebook wrote.

In [ ]:
%%writefile /content/drive/MyDrive/techqa_rq1/example_cases.py
"""
example_cases.py - one real worked example behind every result and every figure.

Each function takes a single question (or a single answer) from the saved result files and
walks it through the calculation the surrounding table or figure aggregates, so a reader can
see how one row becomes the number that is reported. Nothing here computes a new result: every
value is read back from the files the notebooks wrote.

    chunking_example      what each chunking method leaves of one question's gold span
    retrieval_example     one question under the four retrievers
    reranking_example     one question with and without the cross-encoder
    effects_example       one main-effect contrast, recomputed from the paired column
    generation_example    one generated answer, its citations and its correctness check
    attribution_example   one question through the four stage tests, in pipeline order
    prompt_example        the same question under both prompts, with its stage label fixed
    certainty_example     one question's signals, its cross-fitted score and its band
    citation_example      one answer's citation pattern and the band it earns
    rq1_scenario          RQ1 answered from a single question
    rq2_scenario          RQ2 answered from a single question
    rq3_scenario          RQ3 answered from the two ends of the certainty scale

Every function prints a short block and returns the row it used, so a notebook cell can show
the example directly under the result it explains.

    import example_cases as EX
    EX.chunking_example(RESULTS)
"""
from __future__ import annotations

from pathlib import Path

import pandas as pd

WIDTH = 78
CENTRE_METHOD = "sliding_window"


def _rule(title):
    print("\n" + "-" * WIDTH)
    print(f"WORKED EXAMPLE  |  {title}")
    print("-" * WIDTH)


def _read(results, name):
    p = Path(results) / name
    if not p.exists():
        raise FileNotFoundError(f"{p} is missing; run the notebook that writes it first")
    return pd.read_csv(p)


def _pick(frame, prefer=None):
    """A stable choice: the named question if present, else the first by identifier."""
    if prefer is not None and (frame.question_id == prefer).any():
        return prefer
    return sorted(frame.question_id.unique())[0]


# ---------------------------------------------------------------------------------
# RQ1: the three factors
# ---------------------------------------------------------------------------------

def chunking_example(results, question_id=None):
    """One question seen by all five chunking methods, with the centre point first."""
    f = _read(results, "rq1_factorA_per_question.csv")
    # a question the methods disagree about makes the sharpest example, chosen over the four
    # methods that share BGE-small so the difference is segmentation and not the encoder
    same_encoder = f[~f.method.isin(["late_chunking", "sliding_window_longctx"])]
    spread = (same_encoder.groupby("question_id").gold_span_coverage
              .agg(lambda s: s.max() - s.min()).sort_values(ascending=False))
    qid = question_id or (spread.index[0] if len(spread) else _pick(f))
    rows = f[f.question_id == qid].set_index("method")
    _rule(f"chunking method, question {qid}")
    print("How much of this question's gold span each segmentation leaves in the context,")
    print("with the same retriever and the same context depth:\n")
    for method in rows.index:
        r = rows.loc[method]
        print(f"  {method:<24} coverage {float(r.gold_span_coverage):.3f}   "
              f"nDCG@5 {float(r['nDCG@5']):.3f}   "
              f"gold document in context: {'yes' if float(r.gold_document_in_top_5) else 'no'}")
    best, worst = rows.gold_span_coverage.idxmax(), rows.gold_span_coverage.idxmin()
    print(f"\n  The spread on this one question, {float(rows.gold_span_coverage.max()):.3f} under "
          f"{best} against\n  {float(rows.gold_span_coverage.min()):.3f} under {worst}, is one "
          "row of the average the table reports.")
    return rows


def retrieval_example(results, question_id=None):
    f = _read(results, "rq1_factorB_per_question.csv")
    qid = question_id or _pick(f)
    rows = f[f.question_id == qid].set_index("retriever")
    _rule(f"retrieval strategy, question {qid}")
    print("The same question and the same segmentation, scored by each retriever:\n")
    for r_ in rows.index:
        r = rows.loc[r_]
        rank = r.gold_doc_rank
        print(f"  {r_:<18} gold document rank {('-' if pd.isna(rank) else int(rank)):>3}   "
              f"Recall@20 {float(r['Recall@20']):.0f}   coverage {float(r.gold_span_coverage):.3f}")
    print("\n  Recall@20 counts only whether the document arrived; coverage says how much of")
    print("  the answer came with it, which is why both are reported.")
    return rows


def reranking_example(results, question_id=None):
    f = _read(results, "rq1_factorC_per_question.csv")
    changed = f.pivot_table(index="question_id", columns="rerank_level",
                            values="gold_span_coverage", aggfunc="first").dropna()
    if {"off", "cross_encoder"} <= set(changed.columns):
        moved = changed[changed.off != changed.cross_encoder]
        qid = question_id or (moved.index[0] if len(moved) else _pick(f))
    else:
        qid = question_id or _pick(f)
    rows = f[f.question_id == qid].set_index("rerank_level")
    _rule(f"reranking, question {qid}")
    print("The candidate list is identical; only the order in which it is cut to five differs:\n")
    for level in rows.index:
        r = rows.loc[level]
        print(f"  reranking {level:<14} gold document in context: "
              f"{'yes' if float(r.gold_document_in_top_5) else 'no ':<3}   "
              f"coverage {float(r.gold_span_coverage):.3f}   "
              f"context precision {float(r['context_precision@5']):.3f}")
    return rows


def effects_example(results, measure="gold_span_coverage", level="semantic_breakpoint"):
    """The contrast behind one cell of the main-effects table, recomputed from the rows."""
    f = _read(results, "rq1_factorA_per_question.csv")
    a = f[f.method == CENTRE_METHOD].set_index("question_id")[measure]
    b = f[f.method == level].set_index("question_id")[measure]
    both = pd.concat([a.rename("centre"), b.rename("level")], axis=1).dropna()
    diff = both.level - both.centre
    _rule(f"main effect, {level} against the centre point on {measure}")
    print(f"  questions paired            {len(both)}")
    print(f"  mean under {CENTRE_METHOD:<16} {both.centre.mean():.4f}")
    print(f"  mean under {level:<16} {both.level.mean():.4f}")
    print(f"  paired difference           {diff.mean():+.4f}")
    print(f"  questions worse / unchanged / better  "
          f"{int((diff < 0).sum())} / {int((diff == 0).sum())} / {int((diff > 0).sum())}")
    worst = diff.idxmin()
    print(f"\n  The largest single loss is {worst}: {both.loc[worst, 'centre']:.3f} -> "
          f"{both.loc[worst, 'level']:.3f}.")
    print("  The table reports the mean of this column with a bootstrap interval; the figure")
    print("  plots it. This is the column itself.")
    return both


# ---------------------------------------------------------------------------------
# generation and attribution
# ---------------------------------------------------------------------------------

def generation_example(results, answers="rq1_generation_answers.csv",
                       scored="rq1_generation_scored.csv", config_id=None, question_id=None,
                       chars=420):
    """One generated answer: what it cited, what it scored and whether the check passed."""
    a = _read(results, answers)
    s = _read(results, scored)
    cfg = config_id or sorted(a.config_id.unique())[0]
    sub = a[(a.config_id == cfg) & a.answerable.astype(str).str.lower().isin(["true", "1"])]
    qid = question_id or _pick(sub)
    row = sub[sub.question_id == qid].iloc[0]
    sc = s[(s.question_id == qid) & (s.config_id == cfg)]
    _rule(f"one generated answer, question {qid}, configuration {cfg}")
    print(f"  segments given to the model : {row.segment_ids}")
    text = str(row.answer).strip().replace("\n", "\n    ")
    print(f"  answer:\n    {text[:chars]}{'...' if len(text) > chars else ''}")
    if len(sc):
        r = sc.iloc[0]
        print(f"\n  cited segments {int(r.n_cited)}   groundedness {float(r.groundedness):.3f}   "
              f"faithfulness {float(r.faithfulness):.3f}")
        print(f"  token F1 {float(r.token_f1):.3f}   abstained: "
              f"{'yes' if bool(r.abstained) else 'no'}")
        print(f"  the correctness check of Section 5.6 passes at token F1 >= 0.15, so this "
              f"answer {'passes' if float(r.token_f1) >= 0.15 else 'fails'}.")
    return row


def attribution_example(results, question_id=None, configuration=None, threshold=0.80):
    """One question walked through the four tests, in the order the pipeline runs them."""
    import rq2_core as R2
    frames = R2.load_traces(results)
    ceiling = R2.load_ceiling(results)
    table, _ = R2.configuration_table(frames, ceiling)
    cfg = configuration or R2.CENTRE
    sub = table[table.configuration == cfg]
    stages = R2.attribute(sub, threshold)
    qid = question_id or _pick(sub[stages.values != "R"]) if (stages.values != "R").any() \
        else _pick(sub)
    row = sub[sub.question_id == qid].iloc[0]
    stage = R2.attribute(sub[sub.question_id == qid], threshold).iloc[0]
    ceil = float(row[R2.CEILING])
    in20 = str(row.gold_document_in_top_20).lower() in ("1", "1.0", "true", "yes")
    in5 = str(row.gold_document_in_top_5).lower() in ("1", "1.0", "true", "yes")
    cov = float(row.gold_span_coverage)
    _rule(f"stage attribution, question {qid}, {cfg}")
    tests = [
        ("1  does the segmentation still allow 80% of the span?",
         f"ceiling {ceil:.3f}", ceil >= threshold, "S1 segmentation loss"),
        ("2  is the gold Technote among the top-20 candidates?",
         "yes" if in20 else "no", in20, "S2 retrieval miss"),
        ("3  do the chosen segments carry 80% of the span?",
         f"in context: {'yes' if in5 else 'no'}, coverage {cov:.3f}",
         in5 and cov >= threshold, "S3 context exclusion"),
    ]
    for label, observed, passed, fail_label in tests:
        mark = "pass" if passed else "FAIL"
        print(f"  {label:<52} {observed:<28} {mark}")
        if not passed:
            print(f"\n  -> {fail_label}. No test below this one is read.")
            break
    else:
        print(f"  {'4  the evidence was delivered':<52} {'reachable':<28} pass")
        print("\n  -> any failure here is S4 generation drift, which needs a judged answer.")
    print(f"\n  the rule returns: {stage}")
    return row


def prompt_example(results, question_id=None):
    """The same question under both prompts, with every stage label unchanged."""
    o = _read(Path(results) / "rq2", "rq2_generation_outcomes.csv")
    sub = o[o.configuration == o.configuration.iloc[0]]
    qid = question_id or _pick(sub)
    row = sub[sub.question_id == qid].iloc[0]
    _rule(f"prompt comparison, question {qid}")
    print(f"  attributed stage : {row.stage}   (identical under both prompts: retrieval did "
          "not change)")
    print(f"  second run       : abstained {bool(row.abstained)}, correct {bool(row.correct)}, "
          f"outcome {row.outcome}")
    print("\n  Figure 5.12 aggregates exactly this: the stage label is fixed by retrieval, and")
    print("  only the answer column moves when the prompt changes.")
    return row


# ---------------------------------------------------------------------------------
# RQ3: certainty
# ---------------------------------------------------------------------------------

def certainty_example(rq3_dir, question_id=None, split="dev"):
    """One question's signals, the score they produce and the band it lands in."""
    import json
    import rq3_core as R3
    sig = _read(rq3_dir, "rq3_signals.csv")
    res = json.loads((Path(rq3_dir) / "rq3_results.json").read_text(encoding="utf-8"))
    features = res["features"]
    scores = R3.cross_fitted_scores(sig, features)
    sub = sig[sig.split == split]
    qid = question_id or _pick(sub)
    row = sig[sig.question_id == qid].iloc[0]
    p = float(scores.loc[scores.question_id == qid, "score"].iloc[0])
    band = next(b for c, b in zip(list(R3.BAND_CUTS) + [1.01], R3.BANDS) if p < c)
    _rule(f"certainty score, question {qid} ({split} split)")
    print("  the signals a deployed system could compute for this question:")
    for f in features:
        print(f"    {R3.SIGNAL_NAMES.get(f, f):<38} {float(row[f]):+.4f}")
    print(f"\n  score from the model fitted on the other split : {p:.3f}")
    print(f"  band                                           : {band}")
    print(f"  what actually happened                         : evidence "
          f"{'reached' if float(row.label_evidence) else 'did not reach'} the generator")
    print("  (which stage blocked it is RQ2's rule, not this score's: see attribution_example)")
    return row


def citation_example(rq3_dir, band="high"):
    """One answer's citation pattern, the band it earns and whether it passed the check."""
    path = Path(rq3_dir) / "rq3_citation_test_answers.csv"
    if not path.exists():
        _rule("citation band")
        print("  The out-of-sample citation test has not been run in this workspace, so there")
        print("  is no answer to show. It needs notebook D's answers; run the cell headed")
        print("  'The citation rule on answers it was not derived from' above, then this one.")
        return None
    a = _read(rq3_dir, "rq3_citation_test_answers.csv")
    sub = a[a.band == band]
    row = sub.iloc[0]
    _rule(f"citation band, one {band}-band answer")
    print(f"  question {row.question_id}, configuration {row.config_id}")
    print(f"  cited segments {int(row.n_cited)}   cites the top-ranked document: "
          f"{'yes' if bool(row.cites_top_document) else 'no'}   abstained: "
          f"{'yes' if bool(row.abstained) else 'no'}")
    print(f"  -> band {row.band}")
    print(f"  token F1 {float(row.token_f1):.3f}, so the correctness check "
          f"{'passes' if bool(row.correct) else 'fails'}.")
    print("\n  Table 5.15 counts these outcomes band by band over the 150 questions the rule")
    print("  was not derived from.")
    return row


# ---------------------------------------------------------------------------------
# One scenario per research question: a single question, followed from end to end, so the
# answer to the question can be read off a case instead of an average.
# ---------------------------------------------------------------------------------

def rq1_scenario(results, question_id=None):
    """RQ1: which component decides whether the evidence arrives?"""
    f = _read(results, "rq1_factorA_per_question.csv")
    b = _read(results, "rq1_factorB_per_question.csv")
    same_encoder = f[~f.method.isin(["late_chunking", "sliding_window_longctx"])]
    spread = (same_encoder.groupby("question_id").gold_span_coverage
              .agg(lambda s: s.max() - s.min()).sort_values(ascending=False))
    qid = question_id or spread.index[0]
    ch = same_encoder[same_encoder.question_id == qid].set_index("method")
    re_ = b[b.question_id == qid].set_index("retriever")
    _rule(f"RQ1 scenario: question {qid}")
    print("RQ1 asks which component most decides the quality of the evidence the generator\n"
          "receives. Watch one question travel the pipeline twice.\n")
    print("  Changing the SEGMENTATION, retriever fixed:")
    for m in ch.index:
        print(f"    {m:<24} the context holds {float(ch.loc[m].gold_span_coverage):.0%} "
              f"of the gold answer")
    print("\n  Changing the RETRIEVER, segmentation fixed:")
    for r_ in re_.index:
        rank = re_.loc[r_].gold_doc_rank
        print(f"    {r_:<24} gold document rank "
              f"{'not retrieved' if pd.isna(rank) else int(rank):<14} "
              f"context holds {float(re_.loc[r_].gold_span_coverage):.0%}")
    print("\n  Read together: the retriever decides whether the right Technote arrives at all,")
    print("  and the segmentation decides how much of the answer arrives with it. That is the")
    print("  split Section 5.5 reports over all 610 questions, and why RQ1 is answered by a")
    print("  division of labour rather than by a single winning component.")
    return ch


def rq2_scenario(results, question_id=None, configuration=None):
    """RQ2: what does a stage label say that an aggregate score cannot?"""
    import rq2_core as R2
    frames = R2.load_traces(results)
    ceiling = R2.load_ceiling(results)
    table, _ = R2.configuration_table(frames, ceiling)
    cfg = configuration or R2.CENTRE
    sub = table[table.configuration == cfg].reset_index(drop=True)
    stages = R2.attribute(sub)
    blocked = sub[stages.values != "R"]
    qid = question_id or _pick(blocked if len(blocked) else sub)
    _rule(f"RQ2 scenario: question {qid}, {cfg}")
    print("RQ2 asks whether a failed answer can be blamed on one stage, and whether that\n"
          "changes what an engineer would do next.\n")
    attribution_example(results, question_id=qid, configuration=cfg)
    shares = stages.value_counts(normalize=True)
    reach = float(shares.get("R", 0.0))
    print(f"\n  An aggregate score would record one more wrong answer and stop there. The rule")
    print(f"  says which stage owes the fix, and counting the labels over all "
          f"{len(sub)} questions")
    print(f"  puts a ceiling on the generator: {reach:.1%} of questions have their evidence")
    print(f"  delivered, so no change to the model alone can do better than that.")
    return sub


def rq3_scenario(rq3_dir, question_id=None):
    """RQ3: can the system tell, without gold labels, when to trust its own answer?"""
    _rule("RQ3 scenario: two questions, one trustworthy and one not")
    print("RQ3 asks whether signals available at query time can sort answers into bands that\n"
          "mean something. Here are the two ends of the scale.\n")
    import json
    import rq3_core as R3
    sig = _read(rq3_dir, "rq3_signals.csv")
    res = json.loads((Path(rq3_dir) / "rq3_results.json").read_text(encoding="utf-8"))
    scores = R3.cross_fitted_scores(sig, res["features"]).set_index("question_id")
    dev = sig[sig.split == "dev"].copy()
    dev["score"] = scores.reindex(dev.question_id).score.to_numpy()
    top = dev.sort_values("score").dropna(subset=["score"])
    for label, row in (("lowest score", top.iloc[0]), ("highest score", top.iloc[-1])):
        band = next(b for c, b in zip(list(R3.BAND_CUTS) + [1.01], R3.BANDS)
                    if float(row.score) < c)
        print(f"  {label:<14} question {row.question_id}: score {float(row.score):.2f} "
              f"-> band {band}")
        print(f"    top segment stands {float(row.dense_z_top1):+.1f} SD above the rest of the "
              f"pool; dense and lexical retrieval agree on "
              f"{float(row.agree_top5):.0%} of the context")
        print(f"    what actually happened: the evidence "
              f"{'reached' if float(row.label_evidence) else 'did not reach'} the generator\n")
    print("  The band is computed with no gold annotation, which is what makes it usable by a")
    print("  system that has none: it decides when to answer and when to hold back. Section 5.7")
    print("  reports how often that decision is right on questions the model never saw.")
    return dev


ALL = {"chunking": chunking_example, "retrieval": retrieval_example,
       "reranking": reranking_example, "effects": effects_example,
       "generation": generation_example, "attribution": attribution_example,
       "prompt": prompt_example, "certainty": certainty_example,
       "citation": citation_example, "rq1": rq1_scenario, "rq2": rq2_scenario,
       "rq3": rq3_scenario}


In [ ]:
import sys, importlib
if '/content/drive/MyDrive/techqa_rq1' not in sys.path:
    sys.path.insert(0, '/content/drive/MyDrive/techqa_rq1')
import example_cases as EX
EX = importlib.reload(EX)

In [ ]:
_ = EX.reranking_example(C.PATHS.OUT)